In [1]:
from transformers import BertTokenizer, BertForMaskedLM, Trainer, TrainingArguments, GPT2LMHeadModel

In [2]:
import numpy as np

with open("dataset/Sherlock Holmes.txt", "r", encoding="utf-8") as f:
    text = f.read()
print(text[:200])





                        THE ADVENTURES OF SHERLOCK HOLMES

                               Arthur Conan Doyle



                                Table of contents

               A Scandal in Bohem


In [3]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
# model = BertForMaskedLM.from_pretrained('bert-base-uncased')
model = GPT2LMHeadModel.from_pretrained("gpt2")


/Users/prabhatks/miniconda3/envs/cuda/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [4]:
from datasets import Dataset

In [5]:
dataset = Dataset.from_text("dataset/Sherlock Holmes.txt")

In [6]:
dataset

Dataset({
    features: ['text'],
    num_rows: 12195
})

In [7]:
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=28)

In [8]:
train_test_split = dataset.train_test_split(test_size=0.1)  # Split 10% for validation

train_dataset = train_test_split['train']
validation_dataset = train_test_split['test']

# Verify the sizes of the splits
print(train_dataset)
print(validation_dataset)

Dataset({
    features: ['text'],
    num_rows: 10975
})
Dataset({
    features: ['text'],
    num_rows: 1220
})


In [9]:
validation_test_split = validation_dataset.train_test_split(test_size=0.5)

validation_dataset = validation_test_split['train']
test_dataset = validation_test_split['test']

# Verify the sizes of the splits
print(validation_dataset)
print(test_dataset)

Dataset({
    features: ['text'],
    num_rows: 610
})
Dataset({
    features: ['text'],
    num_rows: 610
})


In [10]:
train_dataset = train_dataset.map(tokenize_function, batched=True)
validation_dataset = validation_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/10975 [00:00<?, ? examples/s]

Map:   0%|          | 0/610 [00:00<?, ? examples/s]

Map:   0%|          | 0/610 [00:00<?, ? examples/s]

In [11]:
def add_labels(example):
    # For masked language modeling, labels are the same as input_ids, but we mask some tokens
    example['labels'] = example['input_ids'].copy()  # Copy input_ids to labels
    
    # You may want to add masking logic here if you're doing masked language modeling
    # For simplicity, we're not applying any masking here
    return example

In [12]:
train_dataset = train_dataset.map(add_labels, batched=True)
train_dataset

Map:   0%|          | 0/10975 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 10975
})

In [13]:
validation_dataset = validation_dataset.map(add_labels, batched=True)
validation_dataset

Map:   0%|          | 0/610 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 610
})

In [14]:
training_args = TrainingArguments(
    output_dir='./results',
    evaluation_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    weight_decay=0.01,
    remove_unused_columns=False,  # Prevent automatic removal of unused columns
)

# Setup the Trainer with the new splits
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,   # Use the new train set
    eval_dataset=validation_dataset,  # Use the new validation set
)

/Users/prabhatks/miniconda3/envs/cuda/lib/python3.12/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [15]:
# trainer.train() # going for 7 hours

In [16]:
# input_text = "Once upon a time"
# input_ids = tokenizer.encode(input_text)
# input_ids

In [17]:
# output = model.generate(np.array(input_ids), max_length=50, num_return_sequences=1)
# decoded_output = tokenizer.decode(output[0], skip_special_tokens=True)
# print(decoded_output)